In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc pandas qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# 量子ポートフォリオオプティマイザー: Global Data QuantumによるQiskit Function


*[APIリファレンス](https://docs.quantum.ibm.com/api/functions/global-data-quantum-optimizer)を参照してください。*

> **Note:** Qiskit Functionsは、IBM Quantum&reg; Premiumプラン、Flexプラン、およびOn-Prem（IBM Quantum Platform API経由）プランのユーザーのみが利用できる実験的な機能です。プレビューリリースの状態であり、変更される場合があります。
## 概要
量子ポートフォリオオプティマイザーは、動的ポートフォリオ最適化問題に取り組むQiskit Functionです。これは、リターンを最大化しリスクを最小化するために、一連の資産にわたる定期的な投資のリバランスを目的とする、金融における標準的な問題です。最先端の量子最適化技術を活用することで、この関数はプロセスを簡素化し、量子コンピューティングの専門知識がないユーザーでも、最適な投資トラジェクトリーを見つけるうえでその利点を活用できるようにします。ポートフォリオマネージャー、定量的金融の研究者、および個人投資家に最適で、このツールはポートフォリオ最適化における取引戦略のバックテストを可能にします。
## 関数の説明
量子ポートフォリオオプティマイザー関数は、変分量子固有値ソルバー（VQE）アルゴリズムを使用して二次制約なし二値最適化（QUBO）問題を解き、動的ポートフォリオ最適化問題に対処します。ユーザーは資産価格データを提供し、投資制約を定義するだけで、関数が量子最適化プロセスを実行し、最適化された投資トラジェクトリーのセットを返します。

このプロセスは4つの主要なステージで構成されています。まず、入力データを量子互換の問題にマッピングし、動的ポートフォリオ最適化問題のQUBOを構築し、量子演算子（イジング・ハミルトニアン）に変換します。次に、入力問題とVQEアルゴリズムを量子ハードウェア上で実行できるように適応させます。その後、VQEアルゴリズムを量子ハードウェア上で実行し、最後に結果をポストプロセッシングして最適な投資トラジェクトリーを提供します。このシステムにはノイズ対応（[SQD](/guides/qiskit-addons-sqd)ベース）のポストプロセッシングも含まれており、出力の品質を最大化します。

このQiskit FunctionはGlobal Data Quantumによる[公開論文](https://arxiv.org/abs/2412.19150)に基づいています。
![関数のワークフローの可視化](../docs/images/guides/global-data-quantum-optimizer/function_workflow.svg)
# はじめに
[APIキー](https://quantum.cloud.ibm.com/)を使用して認証し、以下のようにQiskit Functionを選択してください。（このコードスニペットは、[アカウントをローカル環境に保存済み](/guides/functions#install-qiskit-functions-catalog-client)であることを前提としています。）

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [ ]:
# Access function
dpo_solver = catalog.load("global-data-quantum/quantum-portfolio-optimizer")

この例では、[8801.T](https://finance.yahoo.com/quote/8801.T)、[CLF](https://finance.yahoo.com/quote/CLF)、[GBPJPY](https://finance.yahoo.com/quote/GBPJPY)、[ITX.MC](https://finance.yahoo.com/quote/ITX.MC)、[META](https://finance.yahoo.com/quote/META)、[TMBMKDE-10Y](https://finance.yahoo.com/quote/TMBMKDE-10Y)、および[XS2239553048](https://finance.yahoo.com/quote/XS2239553048)の資産を使用しています。以下の図は、この例で使用したデータを示しており、2023年1月1日から9月1日までの資産の日次終値の推移を示しています。

この例では、日付全体で均一性を確保するために、非取引日には直前の利用可能な日付の終値を補完しています。選択した資産が取引日の異なる複数の市場に属しているため、データセットの一貫性を標準化するためにこのステップを適用しています。
![資産の過去データの可視化](../docs/images/guides/global-data-quantum-optimizer/assets.avif)
### 2. 問題を定義する
`qubo_settings`辞書でパラメーターを設定して、問題の仕様を定義してください。

In [ ]:
import os
import glob
import pandas as pd


def read_and_join_csv(file_pattern):
    """
    Reads multiple CSV files matching the file pattern and combines them
    into a single DataFrame.

    Parameters:
    file_pattern (str): The pattern to match CSV files.

    Returns:
    pd.DataFrame: Combined DataFrame with data from all CSV files.
    """
    # Find all files matching the pattern
    csv_files = glob.glob(file_pattern)
    # Get the base file names without the .csv extension
    file_names = [os.path.basename(f).replace(".csv", "") for f in csv_files]
    # Read each CSV file into a DataFrame and set the first column as the index
    df_list = [pd.read_csv(f).set_index("Unnamed: 0") for f in csv_files]

    # Rename columns in each DataFrame to the base file names
    for df, name in zip(df_list, file_names):
        df.columns = [name]

    # Combine all DataFrames into one by merging them side by side
    combined_df = pd.concat(df_list, axis=1)
    return combined_df


file_pattern = "route/to/folder/with/assets/data/*.csv"
assets = read_and_join_csv(file_pattern).to_dict()

### 3. オプティマイザーとアンザッツの設定を定義する（オプション）
オプションで、オプティマイザーとそのパラメーターの選択、およびプリミティブとその設定の指定を含む、最適化プロセスに関する特定の要件を定義します。

Tailored Ansatzの場合、選択された集団サイズは、この値が安定した効率的な最適化をもたらすことを示す以前の実験に基づいています。

Real Amplitudes Ansatzの場合、``population_size``と回路内の量子ビット数の間の線形関係に従うことができます。経験則として、`real_amplitudes`アンザッツには**最低**``population_size ~ 0.8 * n_qubits``を使用することが推奨されます。

Optimized Real AmplitudesはReal Amplitudesアンザッツよりも優れた最適化性能を発揮することが期待されます。ただし、このアンザッツで最適化する変数の数はReal Amplitudesの場合よりもはるかに速く増加します（[論文](https://arxiv.org/pdf/2412.19150)を参照）。したがって、大規模な問題では、Optimized Real Amplitudesはより多くの回路実行が必要です。Optimized Real Amplitudesは最大100量子ビットを必要とする問題に有効である可能性が高いですが、``population_size``パラメーターを設定する際には注意が必要です。この``population_size``のスケールアップの例として、前の表では84量子ビットの問題でOptimize Real Amplitudesが120の``population_size``を必要とするのに対し、56量子ビットの問題では40の``population_size``で十分であることを示しています。

In [ ]:
qubo_settings = {
    "nt": 4,
    "nq": 4,
    "dt": 30,
    "max_investment": 25,
    "risk_aversion": 1000.0,
    "transaction_fee": 0.01,
    "restriction_coeff": 1.0,
}

特定のアンザッツを選択することもできます。以下では``'Tailored'``アンザッツを使用します。

In [ ]:
optimizer_settings = {
    "de_optimizer_settings": {
        "num_generations": 20,
        "population_size": 90,
        "recombination": 0.4,
        "max_parallel_jobs": 5,
        "max_batchsize": 4,
        "mutation_range": [0.0, 0.25],
    },
    "optimizer": "differential_evolution",
    "primitive_settings": {
        "estimator_shots": 25_000,
        "estimator_precision": None,
        "sampler_shots": 100_000,
    },
}

### 4. 問題を実行する

In [ ]:
ansatz_settings = {
    "ansatz": "tailored",
    "multiple_passmanager": False,
}

### 5. 結果を取得する
関数は、目的関数値に基づいて最小から最大の順に並べられた投資トラジェクトリーを含む辞書を返します（APIリファレンスの[出力](https://docs.quantum.ibm.com/api/functions/global-data-quantum-optimizer#output)セクションを参照）。この結果セットにより、最低コストのトラジェクトリーとその対応する投資評価を特定できます。また、さまざまなトラジェクトリーの分析が可能で、特定のニーズや目標に最も合致するものを選択しやすくなります。この柔軟性により、さまざまな好みやシナリオに合わせた選択が可能です。
まず、プロセス中に見つかった最低の目的コストを達成した結果戦略を提示します。

In [ ]:
dpo_job = dpo_solver.run(
    assets=assets,
    qubo_settings=qubo_settings,
    optimizer_settings=optimizer_settings,
    ansatz_settings=ansatz_settings,
    backend_name="<backend name>",
    previous_session_id=[],
    apply_postprocess=True,
)

### 5. Retrieve results

The function returns a dictionary with the investment trajectories ordered from lowest to highest according to their objective function value (see the [Output](/docs/api/functions/global-data-quantum-optimizer#output) section of the API reference). This set of results allows for the identification of the trajectory with the lowest cost and its corresponding investment evaluations. Additionally, it provides for the analysis of different trajectories, facilitating the selection of those that best align with specific needs or objectives. This flexibility ensures that choices can be tailored to suit a variety of preferences or scenarios.

Begin by presenting the resulting strategy that achieved the lowest objective cost found during the process.

In [ ]:
# Get the results of the job
dpo_result = dpo_job.result()

# Show the solution strategy
dpo_result["result"]

{'time_step_0': {'8801.T': 0.11764705882352941,
  'ITX.MC': 0.20588235294117646,
  'META': 0.38235294117647056,
  'GBPJPY=X': 0.058823529411764705,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.058823529411764705,
  'XS2239553048': 0.17647058823529413},
 'time_step_1': {'8801.T': 0.11428571428571428,
  'ITX.MC': 0.14285714285714285,
  'META': 0.2,
  'GBPJPY=X': 0.02857142857142857,
  'TMBMKDE-10Y': 0.42857142857142855,
  'CLF': 0.0,
  'XS2239553048': 0.08571428571428572},
 'time_step_2': {'8801.T': 0.0,
  'ITX.MC': 0.09375,
  'META': 0.3125,
  'GBPJPY=X': 0.34375,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.25},
 'time_step_3': {'8801.T': 0.3939393939393939,
  'ITX.MC': 0.09090909090909091,
  'META': 0.12121212121212122,
  'GBPJPY=X': 0.18181818181818182,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.21212121212121213}}

その後、メタデータを使用して、サンプリングされたすべての戦略の結果にアクセスできます。これにより、オプティマイザーが返した代替トラジェクトリーをさらに分析することができます。これを行うには、`dpo_result['metadata']['all_samples_metrics']`に格納されている辞書を読み込んでください。この辞書には、最適戦略に関する追加情報だけでなく、最適化中に評価された他の候補戦略の詳細も含まれています。

以下の例では、`pandas`を使用してこの情報を読み取り、最適戦略に関連する主要なメトリクスを抽出する方法を示しています。これらには、制約偏差、シャープ比、および対応する投資リターンが含まれます。

In [ ]:
# Convert metadata to a DataFrame
df = pd.DataFrame(dpo_result["metadata"]["all_samples_metrics"])

# Find the minimum objective cost
min_cost = df["objective_costs"].min()
print(f"Minimum Objective Cost Found: {min_cost:.2f}")

# Extract the row with the lowest cost
best_row = df[df["objective_costs"] == min_cost].iloc[0]

# Display the results associated with the best solution
print("Best Solution:")
print(f"  - Restriction Deviation: {best_row['rest_breaches']}%")
print(f"  - Sharpe Ratio: {best_row['sharpe_ratios']:.2f}")
print(f"  - Return: {best_row['returns']}")

Minimum Objective Cost Found: -3.78
Best Solution:
  - Restriction Deviation: 40.0
  - Sharpe Ratio: 24.82
  - Return: 0.46


### 6. Performance analysis

Last, analyze the performance of your optimization application. Specifically, compare your results, obtained in the previous example, against a random baseline to assess the effectiveness of our approach. If the quantum algorithm demonstrably and consistently produces results with lower cost values, it indicates an effective optimization process.

The figure presents the probability distributions of the objective costs. To generate these distributions, take the list of objective costs from the function result and count the occurrences of each cost value (values rounded to the second decimal place). Then, update the count column accordingly by joining counts of identical rounded values. Note that, for better visual comparison, the occurrence counts have been normalized so that each distribution is displayed between 0 and 1.

![Visualization of the solution of the optimization](../docs/images/guides/global-data-quantum-optimizer/cost_distribution.svg)

As shown in the figure (blue solid line), the cost distribution for our Variational Quantum Eigensolver (post-processed with SQD) approach is sharply concentrated at lower objective cost values, indicating good optimization performance. In contrast, the noisy baseline exhibits a broader distribution, centered around higher cost values. The gray dashed vertical line represents the mean value of the random distribution, further highlighting the consistency of the function in returning optimized investment strategies. For an additional comparison, the black dashed line in the figure corresponds to the solution obtained with the Gurobi optimizer (free version). All these results are further explored in the benchmarks below for the "Mixed Assets" example evaluated with the "Tailored" ansatz.

# Benchmarks

This function was tested under different configurations of resolution qubits, ansatz circuits, and groupings of assets from various sectors: a mix of different assets (Set 1), oil derivatives (Set 2), and IBEX35 (Set 3). See more details in the following table.

| Set       | Date       | Assets                                                                 |
|-----------|------------|------------------------------------------------------------------------|
| **Set 1** | 01/01/2023 | 8801.T, CL=F, GBPJPY=X, ITX.MC, META, TMBMKDE-10Y, XS2239553048         |
| **Set 2** | 01/06/2023 | CL=F, BZ=F, HO=F, NG=F, XOM, RB=F, 2222.SR                               |
| **Set 3** | 01/11/2022 | ACS.MC, ITX.MC, FER.MC, ELE.MC, SCYR.MC, AENA.MC, AMS.MC                |

Two key metrics were used to evaluate solution quality.
1. The objective cost, which measures optimization efficiency by comparing the cost function value from each experiment with results from Gurobi (free version).
2. The Sharpe ratio, which captures the risk-adjusted return of each portfolio, offering insight into the financial performance of the solutions.

Together, these metrics benchmark both computational and financial aspects of the quantum-generated portfolios.


| Example             | Qubits | Ansatz                  | Depth | Runtime Usage (s) | Total usage (s) | Objective cost | Sharpe | Gurobi objective cost | Gurobi Sharpe |
|-------------------------------|--------|-------------------------------|--------|-------------------|------------------|----------------|--------|------------------------|----------------|
| Mixed Assets (Set 1, 4 time steps, 4-bit)   | 112    | Tailored                      | 83     | 12735             | 13095            | -3.78          | 24.82  | -4.25                  | 24.71          |
| Mixed Assets (Set 1,4 time steps, 4 time steps, 4-bit)   | 112    | Real Amplitudes               | 359    | 11739             | 11903            | -3.39          | 23.64  | -4.25                  | 24.71          |
| Oil Derivatives (Set 2, 4 time steps, 3-bit)| 84     | Optimized Real Amplitudes     | 78     | 6180              | 6350             | -3.73          | 19.13  | -4.19                  | 21.71          |
| IBEX35 (Set 3, 4 time steps, 2-bit)         | 56     | Optimized Real Amplitudes     | 96     | 3314              | 3523             | -3.67          | 14.48  | -4.11                  | 16.44          |



Results show that the quantum optimizer, with problem-specific ansatzes, effectively identifies efficient investment strategies across various portfolio types.

Below we detail both the population size and the number of generations specified in the `optimizer_options` dictionary. All other parameters were set to their default values.

| **Example**                | ``population_size`` | ``num_generations``   |
|----------------------------|----------------------|----------------------|
| Mixed Assets Portfolio     | 90                   | 20                   |
| Mixed Assets Portfolio     | 92                   | 20                   |
| Oil Derivatives Portfolio  | 120                  | 20                   |
| IBEX35 Portfolio           | 40                   | 20                   |

The number of generations was set to 20, as this value was found to be sufficient to reach convergence. Additionally, the default values for the optimizer's internal parameters were left unchanged, as they consistently provided good performance and are generally recommended by the literature and implementation guidelines.

# Get support
If you need help, you can send an email to qpo.support@globaldataquantum.com. In your message, provide the function job ID.

## Next steps

<Admonition type="note" title="Recommendations">
*   Read [the associated research paper](https://arxiv.org/pdf/2412.19150).
* - Visit the [API reference](/docs/api/functions/global-data-quantum-optimizer) for this Qiskit Function.
*   Request access to the function by filling in [this form](https://globaldatum.io/qiskit-function/#form).
*   Try the [Dynamic Portfolio Optimization](/docs/tutorials/global-data-quantum-optimizer) tutorial.

</Admonition>